In [ ]:
# 1. SETUP Y DEPENDENCIAS

import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Agregar src a path para importar módulos
sys.path.insert(0, '/home/honorio/IA/rnn/src')

# Verificar GPU disponible
import tensorflow as tf
print("TensorFlow version:", tf.__version__)
print("GPU disponible:", tf.config.list_physical_devices('GPU'))
print("Dispositivos:", tf.config.list_physical_devices())

# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Crear estructura de directorios si no existe
base_path = Path('/home/honorio/IA/rnn')
data_raw_path = base_path / 'data' / 'raw'
data_processed_path = base_path / 'data' / 'processed'
splits_path = base_path / 'data' / 'splits'

for path in [data_raw_path, data_processed_path, splits_path]:
    path.mkdir(parents=True, exist_ok=True)
    print(f"✓ Directorio listo: {path}")


# Traductor Automático RNN - Exploración y Preparación de Datos

**Proyecto:** Traductor Español-Inglés con CRISP-ML(Q)  
**Objetivo:** BLEU Score ≥ 0.35  
**Hardware:** GPU ≤ 6GB (optimización crítica)

Este notebook realiza el análisis exploratorio (EDA) y preparación inicial de datos para entrenar un modelo Seq2Seq ligero.

## 2. Crear o Descargar Dataset de Traducción

Para este proyecto usaremos **pares Español-Inglés** del dataset Tatoeba o generaremos un pequeño dataset de ejemplo para demostración.

In [ ]:
# 2a. Crear dataset de ejemplo (pequeño, para demostración local)
# En producción, descargar de: https://tatoeba.org/en/downloads

# Dataset ejemplo: 100 pares Español-Inglés
example_pairs = [
    ("hola", "hello"),
    ("buenos días", "good morning"),
    ("cómo estás", "how are you"),
    ("me llamo juan", "my name is juan"),
    ("de dónde eres", "where are you from"),
    ("soy de españa", "i am from spain"),
    ("qué hora es", "what time is it"),
    ("necesito ayuda", "i need help"),
    ("por favor", "please"),
    ("gracias", "thank you"),
    ("de nada", "you are welcome"),
    ("adiós", "goodbye"),
    ("hasta luego", "see you later"),
    ("hace buen día", "it is a nice day"),
    ("hace frio", "it is cold"),
    ("me gusta", "i like"),
    ("no me gusta", "i do not like"),
    ("eres muy amable", "you are very kind"),
    ("eso es increíble", "that is incredible"),
    ("no entiendo", "i do not understand"),
] * 5  # Repetir para tener 100 pares

print(f"Dataset de ejemplo: {len(example_pairs)} pares")

# Guardar como CSV (formato simple)
pairs_df = pd.DataFrame(example_pairs, columns=['spanish', 'english'])
csv_path = data_raw_path / 'translation_pairs.csv'
pairs_df.to_csv(csv_path, index=False)
print(f"✓ Guardado en: {csv_path}")
print("\nPrimeros ejemplos:")
print(pairs_df.head(10))


## 3. Análisis Exploratorio (EDA)

In [ ]:
# 3a. Cargar y explorar datos
df = pd.read_csv(csv_path)

print("=== ESTADÍSTICAS BÁSICAS ===")
print(f"Total de pares: {len(df)}")
print(f"Filas nulas: {df.isnull().sum()}")
print(f"\nColumnas: {df.columns.tolist()}")
print(f"\nPrimeras filas:\n{df.head()}")

# Función para contar tokens
def count_tokens(text):
    return len(str(text).lower().split())

# Analizar longitudes
df['spanish_tokens'] = df['spanish'].apply(count_tokens)
df['english_tokens'] = df['english'].apply(count_tokens)

print("\n=== ESTADÍSTICAS DE LONGITUD ===")
print(f"Español - Promedio: {df['spanish_tokens'].mean():.2f}, Max: {df['spanish_tokens'].max()}, Min: {df['spanish_tokens'].min()}")
print(f"Inglés  - Promedio: {df['english_tokens'].mean():.2f}, Max: {df['english_tokens'].max()}, Min: {df['english_tokens'].min()}")


In [ ]:
# 3b. Visualizar distribuciones
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df['spanish_tokens'], bins=20, alpha=0.7, label='Español', color='blue')
axes[0].hist(df['english_tokens'], bins=20, alpha=0.7, label='Inglés', color='red')
axes[0].set_xlabel('Número de tokens')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribución de longitud de oraciones')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Scatter plot
axes[1].scatter(df['spanish_tokens'], df['english_tokens'], alpha=0.6)
axes[1].set_xlabel('Tokens en Español')
axes[1].set_ylabel('Tokens en Inglés')
axes[1].set_title('Relación entre longitudes')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✓ Visualizaciones completadas")


## 4. Preprocesamiento: Limpieza y Normalización

In [ ]:
# 4a. Importar preprocesador y aplicar limpieza
from preprocessing import TextPreprocessor, ParallelDataProcessor

# Crear preprocesadores
preprocessor = TextPreprocessor(vocab_size=5000, max_length=30)

# Función de limpieza manual
def clean_text(text):
    import re
    text = str(text).lower().strip()
    text = re.sub(r'[^a-záéíóúñ\s.,!?;]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Aplicar limpieza
df['spanish_clean'] = df['spanish'].apply(clean_text)
df['english_clean'] = df['english'].apply(clean_text)

print("=== DESPUÉS DE LIMPIEZA ===")
print(df[['spanish', 'spanish_clean', 'english', 'english_clean']].head(10))

# Recalcular tokens después de limpieza
df['spanish_tokens_clean'] = df['spanish_clean'].apply(count_tokens)
df['english_tokens_clean'] = df['english_clean'].apply(count_tokens)

print(f"\nEspañol - Promedio: {df['spanish_tokens_clean'].mean():.2f}")
print(f"Inglés  - Promedio: {df['english_tokens_clean'].mean():.2f}")


## 5. Construir Vocabularios y Tokenización

In [ ]:
# 5a. Construir vocabularios
processor = ParallelDataProcessor(vocab_size=5000, max_length=30)

print("Construyendo vocabularios...")
processor.build_vocabs(
    df['spanish_clean'].tolist(),
    df['english_clean'].tolist()
)

print(f"\nVocabulario Español: {processor.source_processor.vocab_size_actual} palabras")
print(f"Vocabulario Inglés: {processor.target_processor.vocab_size_actual} palabras")

# Guardar vocabularios
processor.save_vocabs(
    data_processed_path / 'vocab_spanish.json',
    data_processed_path / 'vocab_english.json'
)
print("✓ Vocabularios guardados")

# Ejemplo de tokenización
example_es = "hola cómo estás"
example_en = "hello how are you"

tokens_es = processor.source_processor.encode(example_es)
tokens_en = processor.target_processor.encode(example_en)

print(f"\nEjemplo - Español:")
print(f"  Texto: '{example_es}'")
print(f"  Tokens: {tokens_es}")
print(f"  Decodificado: {processor.source_processor.decode(tokens_es)}")

print(f"\nEjemplo - Inglés:")
print(f"  Texto: '{example_en}'")
print(f"  Tokens: {tokens_en}")
print(f"  Decodificado: {processor.target_processor.decode(tokens_en)}")


## 6. Preparar Datos: Train/Val/Test Split

In [ ]:
# 6a. Dividir datos en train/val/test
from sklearn.model_selection import train_test_split

# Crear lista de pares limpios
pairs = list(zip(df['spanish_clean'].tolist(), df['english_clean'].tolist()))

# Split: 80% train, 10% val, 10% test
train_pairs, temp_pairs = train_test_split(pairs, test_size=0.2, random_state=42)
val_pairs, test_pairs = train_test_split(temp_pairs, test_size=0.5, random_state=42)

print(f"Conjunto de entrenamiento: {len(train_pairs)} pares")
print(f"Conjunto de validación: {len(val_pairs)} pares")
print(f"Conjunto de prueba: {len(test_pairs)} pares")

# Crear datasets codificados
def create_encoded_dataset(pairs, processor):
    X = []
    y = []
    for source, target in pairs:
        src_idx, tgt_idx = processor.encode_pair(source, target)
        X.append(src_idx)
        y.append(tgt_idx)
    return np.array(X), np.array(y)

print("\nCodificando datos...")
X_train, y_train = create_encoded_dataset(train_pairs, processor)
X_val, y_val = create_encoded_dataset(val_pairs, processor)
X_test, y_test = create_encoded_dataset(test_pairs, processor)

print(f"✓ X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"✓ X_val shape: {X_val.shape}, y_val shape: {y_val.shape}")
print(f"✓ X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")

# Guardar datasets
np.save(splits_path / 'X_train.npy', X_train)
np.save(splits_path / 'y_train.npy', y_train)
np.save(splits_path / 'X_val.npy', X_val)
np.save(splits_path / 'y_val.npy', y_val)
np.save(splits_path / 'X_test.npy', X_test)
np.save(splits_path / 'y_test.npy', y_test)
print("✓ Datos guardados en data/splits/")


## 7. Siguientes Pasos

✓ **Completado:**
- Estructura de directorios creada
- Dataset de ejemplo cargado
- EDA realizado
- Datos limpios y normalizados
- Vocabularios construidos
- Datasets codificados y guardados

📝 **Próximas acciones:**
1. Crear notebook `02_model_training.ipynb` para entrenar el modelo Seq2Seq
2. Implementar bucle de entrenamiento con early stopping
3. Evaluar con BLEU Score
4. Ajustar hiperparámetros si es necesario
5. Exportar modelo entrenado

**Nota:** Con un dataset pequeño (100 pares), los resultados iniciales serán básicos. Para un modelo robusto, usar un corpus de miles de pares de Tatoeba o OPUS.